# Text classification using shallow machine learning techniques

En esta práctica vamos a utilizar el dataset de SMS labeleados con spam/ham que hemos usado anteriormente para construir un clasificador supervisado.

Para ello, vamos a transformar el texto a dos tipos de features:


*   Bag Of Words (CountVectorizer)
*   TF-IDF (TfidfVectorizer)

Y vamos a probar cada uno de estos features con distintos clasificadores:



*   Logistic Regression
*   Random Forest
*   Decision Tree
*   SVM (lineal y no lineal)





In [ ]:
!wget https://drive.google.com/uc?id=1yCTgTEyj5-M2aoUlA0Vh1IF7cCxZLhdy -O sms_spam.csv

--2024-10-07 16:17:31--  https://drive.google.com/uc?id=1yCTgTEyj5-M2aoUlA0Vh1IF7cCxZLhdy
Resolving drive.google.com (drive.google.com)... 74.125.126.102, 74.125.126.101, 74.125.126.139, ...
Connecting to drive.google.com (drive.google.com)|74.125.126.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1yCTgTEyj5-M2aoUlA0Vh1IF7cCxZLhdy [following]
--2024-10-07 16:17:31--  https://drive.usercontent.google.com/download?id=1yCTgTEyj5-M2aoUlA0Vh1IF7cCxZLhdy
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.193.132, 2607:f8b0:4001:c0f::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.193.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 493232 (482K) [application/octet-stream]
Saving to: ‘sms_spam.csv’

sms_spam.csv        100%[===================>] 481.67K  --.-KB/s    in 0.005s  

2024-10-07 16:17:33 (86.8 MB/s) 

En la siguiente celda procesamos el dataset reutilizando código de una práctica anterior. Por último, dividimos el dataset en traint y test.

In [ ]:
import pandas as pd
from nltk.tokenize.casual import casual_tokenize
from sklearn.model_selection import train_test_split


#Cargamos el CSV con pandas
df = pd.read_csv('sms_spam.csv')

#Vamos a reemplazar el index por defecto del dataframe (0, len) por "smsX" o "smsX!" siendo X la posición en la lista y "!" define que el ejemplo es spam
index = ['sms{}{}'.format(i, '!'*j) for (i, j) in zip(range(len(df)), df.spam)]

sms = pd.DataFrame(df.values, columns=df.columns, index=index)
sms['spam'] = sms.spam.astype(int)

print(f"Hay un total de {len(sms)} sms de los que {sms.spam.sum()} son spam." )
print(sms.head(6))

# Dividimos el dataset en training and testing sets
X = sms['text']
y = sms['spam']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Hay un total de 4837 sms de los que 638 son spam.
      Unnamed: 0  spam                                               text
sms0           0     0  Go until jurong point, crazy.. Available only ...
sms1           1     0                      Ok lar... Joking wif u oni...
sms2!          2     1  Free entry in 2 a wkly comp to win FA Cup fina...
sms3           3     0  U dun say so early hor... U c already then say...
sms4           4     0  Nah I don't think he goes to usf, he lives aro...
sms5!          5     1  FreeMsg Hey there darling it's been 3 week's n...


Ahora vamos a transformar el texto en crudo a Bag of Words utilizando el CountVectorizer de sklearn.

**Ejercicio 1:** Prepara el train y test sets para trabajar con CountVectorizer y TfidfVectorizer.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer(tokenizer=casual_tokenize)
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Por otro lado, también usamos TF-IDF para representar nuestro texto.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(tokenizer=casual_tokenize)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

**Ejercicio 2:** Una vez transformado el texto de entrada en dos representaciones diferentes, evalua su rendimiento con los siguientes clasificadores:

*   Logistic Regression
*   Random Forest
*   Decision Tree
*   SVM (lineal y no lineal)

Para ello, crea un metodo que acepte diferentes train y test sets (una pareja para el texto como bag of words y otro como Tf-Idf).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
import numpy as np

def classify_text(X_train, y_train, X_test, y_test, feature_name, vectorizer):

    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)

    lr = LogisticRegression(random_state=42)
    lr.fit(X_train, y_train)

    rf_predictions = rf.predict(X_test)
    lr_predictions = lr.predict(X_test)

    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train, y_train)

    linear_svm = LinearSVC(random_state=42)
    linear_svm.fit(X_train, y_train)

    nonlinear_svm = SVC(random_state=42)
    nonlinear_svm.fit(X_train, y_train)


    models = {
        "RandomForest": rf,
        "Logistic Regression": lr,
        "Decision Tree": dt,
        "Linear SVM": linear_svm,
        "Non-linear SVM": nonlinear_svm
    }

    for model_name, model in models.items():
        predictions = model.predict(X_test_count)
        print(f"Resultados de {model_name} con el texto representado como {feature_name}:")
        print(classification_report(y_test, predictions))
        print("-" * 60)

    importances = rf.feature_importances_
    feature_names = vectorizer.get_feature_names_out()
    sorted_indices = np.argsort(importances)[::-1]
    print("Top palabras para Random Forest:")
    for i in range(10):
      print(feature_names[sorted_indices[i]], importances[sorted_indices[i]])


    coef = lr.coef_[0]
    sorted_indices = np.argsort(coef)[::-1]
    print("\nTop palabras para Regresión Logística:")
    for i in range(10):
      print(feature_names[sorted_indices[i]], coef[sorted_indices[i]])

Ahora entrenemos y evaluemos los modelos con las dos formas de representar texto.


In [ ]:
classify_text(X_train_count, y_train, X_test_count, y_test, "CountVectorizer", count_vect)
classify_text(X_train_tfidf, y_train, X_test_tfidf, y_test, "TfidfVectorizer", tfidf)

/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Resultados de RandomForest con el texto representado como CountVectorizer:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       842
           1       0.99      0.77      0.87       126

    accuracy                           0.97       968
   macro avg       0.98      0.88      0.92       968
weighted avg       0.97      0.97      0.97       968

------------------------------------------------------------
Resultados de Logistic Regression con el texto representado como CountVectorizer:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       842
           1       0.98      0.83      0.90       126

    accuracy                           0.98       968
   macro avg       0.98      0.92      0.94       968
weighted avg       0.98      0.98      0.98       968

------------------------------------------------------------
Resultados de Decision Tree con el texto representado como CountVec

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Ejercicio 3:** ¿Qué otra forma de representar texto hemos visto? ¿Podríamos usarlo con este tipo de clasificadores?